In [ ]:
import pandas as pd
import numpy as np
import glob
import os
from scipy import stats
from datetime import datetime, timedelta
from typing import Union, List

In [ ]:
# Path to the folder including the csv files
data_folder_path = '/Users/ebrukasikaralar/data_parallel_server/data_july02_oct02/csv_files/'

In [ ]:
def get_file_list(folder_path, file_pattern):
    """
    Generates a list of file paths from a specified folder that match a given pattern.

    Parameters:
    - folder_path: str - Path to the folder where files are located.
    - file_pattern: str - Pattern to match files.

    Returns:
    - list: List of file paths matching the pattern in the specified folder.
    """
    
    if not os.path.exists(folder_path):
        raise FileNotFoundError(f"The specified folder path does not exist: {folder_path}")

    file_list = glob.glob(os.path.join(folder_path, file_pattern))

    if not file_list:
        print(f"No files found in {folder_path} matching the pattern '{file_pattern}'")

    return file_list

In [ ]:
file_list_cust_subcalls = get_file_list(data_folder_path, file_pattern="*_cust_subcalls.csv")

In [ ]:
def filt_abandon(file_path,service,node=0):
    """
    Filters abandonment times from data for a specific service and node.

    Parameters:
    - file_path: str - Path to the data file.
    - service: int - Service type to filter.
    - node: int - Node number to filter, default is 0.

    Returns:
    - DataFrame: Filtered data with abandonment times.
    """
    
    try:
        data = pd.read_csv(file_path)
    except Exception as e:
        raise FileNotFoundError(f"Error reading file {file_path}: {e}")
    
    # Eliminating outliers through distribution plot of queueing and service times
    data = data[(data["queue_time"] < 900) & (data["service_time"] < 1800)]

    # Removing calls with abnormal outcomes
    abnormal_outcomes = [4,13,14,23,30,40,50]
    calls_with_abnormal_outcome = data[data["outcome"].isin(abnormal_outcomes)]
    call_ids_abnormal_outcome = calls_with_abnormal_outcome["call_id"].unique()
    drop_index = data[data["call_id"].isin(call_ids_abnormal_outcome)].index
    data = data.drop(index = drop_index)
    
    # Focusing only on 1st customer subcalls
    data = data[data["cust_subcall"] == 1]
    
    # Sorting and removing duplicates
    data = data.sort_values(["segment_start"])
    data = data.drop_duplicates("call_id", keep = "last")
    
    data = data[data["service"] == service]
    
    # If service is 1, focusing on a specific node
    if service == 1:
        data = data[data["node"] == node]
    
    # Convert segment_start to datetime and adjust for timezone
    data["Datetime"] = pd.to_datetime(data["segment_start"], unit="s", utc=True)
    
    # Filter data between 10 AM and 4 PM
    data = data.set_index('Datetime')
    data = data.between_time('10:00','16:00')
    data = data.dropna(axis = 0)
    
    if data.empty:
        return 0
    
    else:
        data.loc[data["outcome"].isin([11,12]),"abandon"] = 1 # If abandoned the abandonment time is not censored
        data.loc[~data["outcome"].isin([11,12]),"abandon"] = 0 # If not abandoned the abandonment time is censored
    
        return data[["call_id","segment_start","queue_time","abandon"]]

In [ ]:
# Bias corrected Kaplan Meier estimator

def kmi(survival_times, censored_flags): 
    """
    Calculates the Kaplan-Meier Integrator (KMI) for survival times.

    Parameters:
    - survival_times: array-like - An array of survival times.
    - censored_flags: array-like - A binary array indicating whether each survival time is censored (0) or not (1).

    Returns:
    - float: The estimated mean survival time.
    """
    if len(survival_times) != len(censored_flags):
        raise ValueError("Survival times and censored flags must be the same length.")

        
    sorted_indices = np.argsort(survival_times)
    sorted_times = np.sort(survival_times)
    sorted_flags = censored_flags[sorted_indices]
    n = len(sorted_flags)

    km_weights = np.zeros(n)
    km_weights[0] = 1 / n

    for i in range(1, n):
        km_weights[i] = km_weights[i - 1] * (n - (i+1) + 2)/(n - (i+1) + 1) * (((n - (i+1) + 1)/(n - (i+1) + 2))**sorted_flags[i - 1])

    weighted_flags = km_weights * sorted_flags

    if sorted_flags[-1] == 0:
        weighted_flags[-1] = 1 - np.sum(weighted_flags)

    kmi_estimate = np.sum(weighted_flags * sorted_times)
    return kmi_estimate

In [ ]:
def calculate_bias(survival_times, censored_flags):
    """
    Calculates the bias in the Kaplan-Meier estimator.

    Parameters:
    - survival_times: array-like - An array of survival times.
    - censored_flags: array-like - A binary array indicating whether each survival time is censored (0) or not (1).

    Returns:
    - float: The calculated bias.
    """
    if len(survival_times) != len(censored_flags):
        raise ValueError("Survival times and censored flags must be the same length.")

    sorted_indices = np.argsort(survival_times)
    sorted_times = np.sort(survival_times)
    sorted_flags = censored_flags[sorted_indices]
    n = len(sorted_flags)

    bias_factors = np.zeros(n - 2)
    bias_factors[0] = ((n - 2) / (n - 1)) ** sorted_flags[0]

    for i in range(1, n - 2):
        bias_factors[i] = bias_factors[i-1]*(((n - (i+1) - 1)/(n - (i+1)))**sorted_flags[i])

    bias_value = -(n - 1) / n * sorted_times[-1] * sorted_flags[-1] * (1 - sorted_flags[-2]) * bias_factors[-3]
    return bias_value

In [ ]:
def jackknife_estimation(survival_times, censored_flags):
    """
    Calculates the bias-corrected Kaplan-Meier estimator using the jackknife method.

    Parameters:
    - survival_times: array-like - An array of survival times.
    - censored_flags: array-like - A binary array indicating whether each survival time is censored (0) or not (1).

    Returns:
    - float: The bias-corrected Kaplan-Meier estimate.
    """
    kmi_estimate = kmi(survival_times, censored_flags)
    bias_estimate = calculate_bias(survival_times, censored_flags)
    bias_corrected_estimate = kmi_estimate - bias_estimate
    return bias_corrected_estimate

In [ ]:
def km_estimation(file_list, service, node):
    """
    Concatenates data from multiple files to calculate the mean abandonment times
    using a bias-corrected Kaplan-Meier estimator.

    Parameters:
    - file_list: list of str - List of file paths.
    - service: int - Service type to filter.
    - node: int - Node number to filter.

    Returns:
    - float: Bias-corrected Kaplan-Meier estimate of mean abandonment times.
    """
    if not file_list:
        raise ValueError("File list is empty.")

    main_dataframe = pd.DataFrame(filt_abandon(file_list[0],service,node))
    
    for i in range(1,len(file_list)):
                
        data = filt_abandon(file_list[i], service, node)
    
        if isinstance(data, pd.DataFrame):
            df = pd.DataFrame(data)
            main_dataframe = pd.concat([main_dataframe, df], axis=0)
        
        # Check if data is 0 and skip this iteration if it is
        else: 
            continue
    
    # Extracting queue times and abandonment flags
    queue_times = main_dataframe["queue_time"].to_numpy()
    abandonment_flags = main_dataframe["abandon"].to_numpy()

    # Calculating the bias-corrected Kaplan-Meier estimate
    bias_corrected_estimate = jackknife_estimation(queue_times, abandonment_flags)
    
    return bias_corrected_estimate

In [ ]:
# Class names and class indices as they appear in the dataset
class_names = ["Retail_Node1", "Retail_Node2", "Retail_Node3", "Premier", "Business", "Platinum", "Consumer_Loans", "Online_Banking", "EBO", "Telesales", "Subanco", "Case_Quality", "Priority_Service"]

service_class_indices = [1, 1, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11]

# Create a DataFrame to store the abandonment times
abandonment_times = pd.DataFrame(index=class_names, columns=["Total"])

# Calculate and save abandonment times for each class
for i, class_name in enumerate(class_names):
    print(i)
    node = i + 1 if i < 3 else 0  # First three are retail classes with specific nodes
    abandonment_times.loc[class_name, "Total"] = km_estimation(file_list_cust_subcalls, service_class_indices[i], node) #abandonment times in second

In [ ]:
abandonment_rates = (3600 / abandonment_times).round(2)
abandonment_rates